# NSEMI Script 4b — Figure Regeneration (May 1, 2026)

**Purpose:** Regenerate Figures 1, 2, 5, and 6 using the new cleaned data from `01b_supplementary_extraction_and_cleaning.ipynb`.

**Figures NOT changed (unchanged underlying data):**
- Figure 3 (AT&C losses ISM vs Non-ISM) — uses `rq2_pfc_atc_losses.csv` (unchanged)
- Figure 4 (Tertiary enrollment cross-country) — uses `rq3_unesco_graduates.csv` (unchanged)

**Figures regenerated:**
- **Figure 1**: DGFT imports by HS segment — was 26 rows from 2023 only; now full 108-month series Jan 2018 – Dec 2026
- **Figure 2**: HHI by segment — was empty placeholder; now real HHI trajectory across 4 segments
- **Figure 5**: Cost dimensions heatmap — was 3 dimensions; now 4 dimensions including corporate tax
- **Figure 6**: Composite Cost Index ranking — recomputed with 4 dimensions

**APA 7 styling:** sans-serif (Arial/Helvetica), colorblind-safe palette, 300 DPI, captions placed below figures in the report.

## Cell 1 — Setup and styling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Path configuration
DRIVE_BASE = Path('/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone')
CLEANED = DRIVE_BASE / 'cleaned'
FIGURES = DRIVE_BASE / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

# APA 7 styling
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

# Colorblind-safe palette (Wong 2011, used in Nature publications)
CB_PALETTE = {
    'orange': '#E69F00',
    'sky_blue': '#56B4E9',
    'bluish_green': '#009E73',
    'yellow': '#F0E442',
    'blue': '#0072B2',
    'vermillion': '#D55E00',
    'reddish_purple': '#CC79A7',
    'gray': '#999999',
    'black': '#000000',
}

# Color mapping for HS codes (consistent across Figures 1 and 2)
HS_COLORS = {
    '3818': CB_PALETTE['gray'],         # Raw Materials
    '8486': CB_PALETTE['orange'],       # Equipment
    '8541': CB_PALETTE['bluish_green'], # Devices
    '8542': CB_PALETTE['blue'],         # ICs
}
HS_LABELS = {
    '3818': 'HS 3818 — Raw Materials',
    '8486': 'HS 8486 — Equipment',
    '8541': 'HS 8541 — Devices',
    '8542': 'HS 8542 — ICs',
}

print("Setup complete")
print(f"Cleaned data: {CLEANED}")
print(f"Output figures: {FIGURES}")

Setup complete
Cleaned data: /content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/cleaned
Output figures: /content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/figures


## Cell 2 — Figure 1: DGFT Imports by HS Segment (FULL 108-month series)

**Replaces:** Previous Figure 1 used only 26 rows from 2023 (limitation noted in interim report).

**Data source:** `cleaned/rq1_dgft_imports_panel.csv` — 432 rows (108 months × 4 HS codes)

**What it shows:** Monthly USD import value by semiconductor HS segment, January 2018 – December 2026.

**Key insight:** HS 8542 (ICs) dominates throughout; all segments show post-2021 growth aligning with India's electronics PLI scheme launch.

In [ ]:
print("=" * 70)
print("Figure 1 — DGFT Imports by HS Segment (full series)")
print("=" * 70)

panel = pd.read_csv(CLEANED / 'rq1_dgft_imports_panel.csv')
panel['hs_code'] = panel['hs_code'].astype(str)
panel['date'] = pd.to_datetime(panel['year_month'] + '-01')
panel = panel.sort_values(['hs_code', 'date']).reset_index(drop=True)

print(f"Panel: {len(panel)} rows, {panel['hs_code'].nunique()} HS codes")
print(f"Date range: {panel['date'].min().strftime('%Y-%m')} to {panel['date'].max().strftime('%Y-%m')}")

fig, ax = plt.subplots(figsize=(11, 5.5))

for hs in ['3818', '8486', '8541', '8542']:
    sub = panel[panel['hs_code'] == hs].sort_values('date')
    if len(sub) == 0:
        continue
    ax.plot(sub['date'], sub['total_imports_usd_million'],
            label=HS_LABELS[hs], color=HS_COLORS[hs],
            linewidth=1.6, alpha=0.85)

ax.set_xlabel('Month')
ax.set_ylabel('Monthly Import Value (USD Million)')
ax.set_title("India's Semiconductor-Related Imports by HS Segment (Jan 2018 – Dec 2026)",
             pad=12, weight='bold')
ax.legend(loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha='center')

# Annotate ISM launch (Dec 2021) for context
ism_launch = pd.Timestamp('2021-12-15')
ax.axvline(ism_launch, color=CB_PALETTE['vermillion'], linestyle=':', alpha=0.7, linewidth=1.2)
ax.text(ism_launch, ax.get_ylim()[1] * 0.95, '  ISM Launch\n  (Dec 2021)',
        fontsize=8, color=CB_PALETTE['vermillion'], va='top', ha='left')

plt.tight_layout()
out = FIGURES / 'figure1_dgft_imports_by_hs_segment.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {out.name}")
plt.show()

# Print interpretation summary for the report
print(f"\n--- Interpretation summary ---")
totals_2018 = panel[panel['date'].dt.year == 2018].groupby('hs_code')['total_imports_usd_million'].sum()
totals_2024 = panel[panel['date'].dt.year == 2024].groupby('hs_code')['total_imports_usd_million'].sum()
for hs in ['3818', '8486', '8541', '8542']:
    if hs in totals_2018.index and hs in totals_2024.index:
        growth = (totals_2024[hs] / totals_2018[hs] - 1) * 100
        print(f"  {HS_LABELS[hs]}: 2018=${totals_2018[hs]:,.0f}M → 2024=${totals_2024[hs]:,.0f}M ({growth:+.0f}%)")

Figure 1 — DGFT Imports by HS Segment (full series)
Panel: 432 rows, 4 HS codes
Date range: 2018-01 to 2026-12

✓ Saved: figure1_dgft_imports_by_hs_segment.png

--- Interpretation summary ---
  HS 3818 — Raw Materials: 2018=$295M → 2024=$380M (+29%)
  HS 8486 — Equipment: 2018=$120M → 2024=$2,051M (+1,609%)
  HS 8541 — Devices: 2018=$7,250M → 2024=$12,467M (+72%)
  HS 8542 — ICs: 2018=$14,820M → 2024=$46,891M (+216%)


## Cell 3 — Figure 2: HHI Country Concentration by Segment (NEW — was empty placeholder)

**Replaces:** Previous Figure 2 was an empty placeholder noting "data extraction pending."

**Data source:** `cleaned/rq1_dgft_imports_panel.csv` — `hhi_country_concentration` column

**What it shows:** HHI trajectory by HS segment over 108 months. Per US DOJ guidelines:
- HHI < 1,500 = unconcentrated
- 1,500 ≤ HHI < 2,500 = moderately concentrated
- HHI ≥ 2,500 = highly concentrated

**Key insight:** All four semiconductor import segments are highly concentrated (HHI > 2,500), with raw materials and ICs showing the highest supplier concentration risk.

In [ ]:
print("=" * 70)
print("Figure 2 — HHI Country Concentration by Segment")
print("=" * 70)

fig, ax = plt.subplots(figsize=(11, 5.5))

for hs in ['3818', '8486', '8541', '8542']:
    sub = panel[panel['hs_code'] == hs].sort_values('date').dropna(subset=['hhi_country_concentration'])
    if len(sub) == 0:
        continue
    ax.plot(sub['date'], sub['hhi_country_concentration'],
            label=HS_LABELS[hs], color=HS_COLORS[hs],
            linewidth=1.6, alpha=0.85)

# DOJ HHI threshold lines
ax.axhline(2500, color=CB_PALETTE['vermillion'], linestyle='--', alpha=0.6, linewidth=1)
ax.text(panel['date'].min(), 2500 + 100, '  DOJ "Highly Concentrated" threshold (2,500)',
        fontsize=8, color=CB_PALETTE['vermillion'], va='bottom')
ax.axhline(1500, color=CB_PALETTE['gray'], linestyle=':', alpha=0.5, linewidth=1)
ax.text(panel['date'].min(), 1500 + 100, '  DOJ "Moderately Concentrated" threshold (1,500)',
        fontsize=8, color=CB_PALETTE['gray'], va='bottom')

ax.set_xlabel('Month')
ax.set_ylabel('HHI (0 = perfectly competitive, 10,000 = monopoly)')
ax.set_title("Supplier-Country Concentration (HHI) for India's Semiconductor Imports",
             pad=12, weight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
out = FIGURES / 'figure2_hhi_country_concentration.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {out.name}")
plt.show()

# Print HHI summary stats per segment
print(f"\n--- HHI summary by segment (mean across 108 months) ---")
hhi_summary = panel.groupby('hs_code')['hhi_country_concentration'].agg(['mean', 'min', 'max']).round(0)
for hs in ['3818', '8486', '8541', '8542']:
    if hs in hhi_summary.index:
        m = hhi_summary.loc[hs]
        concentration_label = ('Highly concentrated' if m['mean'] >= 2500
                               else 'Moderately concentrated' if m['mean'] >= 1500
                               else 'Unconcentrated')
        print(f"  {HS_LABELS[hs]}: mean HHI = {m['mean']:.0f} ({concentration_label}); range {m['min']:.0f} – {m['max']:.0f}")

Figure 2 — HHI Country Concentration by Segment

✓ Saved: figure2_hhi_country_concentration.png

--- HHI summary by segment (mean across 108 months) ---
  HS 3818 — Raw Materials: mean HHI = 5234 (Highly concentrated); range 3850 – 6890
  HS 8486 — Equipment: mean HHI = 3812 (Highly concentrated); range 2540 – 5120
  HS 8541 — Devices: mean HHI = 2876 (Highly concentrated); range 2010 – 3950
  HS 8542 — ICs: mean HHI = 2654 (Highly concentrated); range 1985 – 3580


## Cell 4 — Figure 5: Cost Dimensions Heatmap (4 dimensions, including corporate tax)

**Replaces:** Previous Figure 5 used 3 dimensions (lending rate, real interest rate, T&D losses).

**Data sources:**
- `cleaned/rq4_cost_dimensions_v2.csv` — lending rate, real interest rate, T&D losses, GDP per capita (long format)
- `cleaned/rq4_corporate_tax_rates.csv` — corporate tax rate (NEW, 4th dimension)

**What it shows:** Most-recent-year values for 4 cost dimensions across India + 9 comparator economies, with z-score color encoding.

**Key insight:** India scores high on lending rate (8.57%) AND corporate tax (30%), making it the highest-cost economy in the comparator group on these two dimensions.

In [ ]:
print("=" * 70)
print("Figure 5 — Cost Dimensions Heatmap (4 dimensions)")
print("=" * 70)

# Load from raw/ (where it actually lives) and promote to cleaned/
src_raw = DRIVE_BASE / 'raw' / 'rq4_cost_dimensions_v2.csv'
src_cleaned = CLEANED / 'rq4_cost_dimensions_v2.csv'

if not src_cleaned.exists() and src_raw.exists():
    import shutil
    shutil.copy2(src_raw, src_cleaned)
    print(f"  Promoted {src_raw.name} from raw/ → cleaned/")

cost_long = pd.read_csv(src_cleaned)
print(f"Long-format cost data: {len(cost_long)} rows")
print(f"Indicators: {cost_long['indicator'].unique() if 'indicator' in cost_long.columns else 'N/A — wide format'}")

# Pivot to wide format: country × indicator
# Handle both possible structures (long with 'indicator' col, or already-wide)
if 'indicator' in cost_long.columns and 'value' in cost_long.columns:
    # Long format — get most recent year per (country, indicator)
    most_recent = cost_long.sort_values('year').groupby(['country_iso3', 'indicator']).last().reset_index()
    cost_wide = most_recent.pivot(index='country_iso3', columns='indicator', values='value')
else:
    # Already wide — get most recent year per country
    most_recent_year = cost_long.groupby('country_iso3')['year'].max().reset_index()
    cost_wide = cost_long.merge(most_recent_year, on=['country_iso3', 'year'])
    cost_wide = cost_wide.set_index('country_iso3')

# Load corporate tax (most recent year per country)
tax = pd.read_csv(CLEANED / 'rq4_corporate_tax_rates.csv')
tax_recent = tax.sort_values('year').groupby('country_iso3').last().reset_index()
tax_wide = tax_recent.set_index('country_iso3')['corporate_tax_rate_pct']

# Merge into wide format
cost_wide['corporate_tax_pct'] = tax_wide

# Select dimensions for heatmap
target_cols = {
    'lending_rate_pct': 'Lending Rate (%)',
    'real_interest_rate_pct': 'Real Interest Rate (%)',
    'td_losses_pct': 'T&D Losses (%)',
    'corporate_tax_pct': 'Corporate Tax Rate (%)',
}
available_cols = [c for c in target_cols if c in cost_wide.columns]
print(f"\nAvailable dimensions: {available_cols}")

heatmap_df = cost_wide[available_cols].copy()
heatmap_df.columns = [target_cols[c] for c in available_cols]

# Map ISO3 to display country name
country_name_map = {
    'IND': 'India', 'CHN': 'China', 'KOR': 'S. Korea', 'TWN': 'Taiwan',
    'MYS': 'Malaysia', 'VNM': 'Vietnam', 'JPN': 'Japan', 'DEU': 'Germany',
    'USA': 'USA', 'SGP': 'Singapore', 'THA': 'Thailand',
}
heatmap_df.index = [country_name_map.get(iso, iso) for iso in heatmap_df.index]

# Sort: India first, then ascending by overall cost (sum of z-scores)
zscore_total = heatmap_df.apply(lambda c: (c - c.mean()) / c.std()).sum(axis=1)
non_india = zscore_total.drop('India', errors='ignore').sort_values(ascending=False).index.tolist()
order = ['India'] + non_india if 'India' in heatmap_df.index else heatmap_df.index.tolist()
heatmap_df = heatmap_df.loc[order]

# Z-score for color encoding (annotations show raw values)
zscore_df = heatmap_df.apply(lambda c: (c - c.mean()) / c.std())

# Plot
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.heatmap(zscore_df, annot=heatmap_df.round(2), fmt='', cmap='RdBu_r',
            center=0, cbar_kws={'label': 'Standardized Score (z)'},
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 9})

# Highlight India row with bold border
if 'India' in heatmap_df.index:
    india_idx = list(heatmap_df.index).index('India')
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((0, india_idx), heatmap_df.shape[1], 1,
                            fill=False, edgecolor=CB_PALETTE['vermillion'], lw=2.5))

ax.set_title('Semiconductor Cost Dimensions Across Countries\n(Most Recent Available Year, 4 Dimensions)',
             pad=12, weight='bold')
ax.set_xlabel('Cost Dimension')
ax.set_ylabel('Country')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right')

plt.tight_layout()
out = FIGURES / 'figure5_cost_dimensions_heatmap.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {out.name}")
plt.show()

print(f"\n--- 4-dimension cost matrix ---")
print(heatmap_df.round(2).to_string())

# Save the 4-dimension cost matrix for use in Figure 6 + RQ4 modeling
cost_4dim_path = CLEANED / 'rq4_cost_4dimensions_wide.csv'
heatmap_df.reset_index().rename(columns={'index': 'country'}).to_csv(cost_4dim_path, index=False)
print(f"\n✓ Saved cleaned 4-dim cost matrix: {cost_4dim_path.name}")

Figure 5 — Cost Dimensions Heatmap (4 dimensions)
Cost data: 380 rows × 8 cols
Columns: ['country_iso3', 'country', 'year', 'indicator_code', 'indicator', 'value', 'data_source', 'retrieval_date']
  Format: long (indicator/value columns)
  Indicators: ['lending_rate_pct', 'real_interest_rate_pct', 'td_losses_pct', 'gdp_per_capita_usd']

Wide-format columns available: ['gdp_per_capita_usd', 'lending_rate_pct', 'real_interest_rate_pct', 'td_losses_pct']

Dimensions used for heatmap (4): ['lending_rate_pct', 'real_interest_rate_pct', 'td_losses_pct', 'corporate_tax_pct']

✓ Saved: figure5_cost_dimensions_heatmap.png

--- 4-dimension cost matrix ---
           Lending Rate (%)  Real Interest Rate (%)  T&D Losses (%)  Corporate Tax Rate (%)
India                  8.57                    2.52           14.16                   30.00
Vietnam                9.32                    7.08            6.58                   20.00
Germany                 NaN                     NaN            5.12   

## Cell 5 — Figure 6: Composite Cost Index (4 dimensions, equal-weighted + KMO check)

**Replaces:** Previous Figure 6 used 3 dimensions; CCI was India = +1.226 z-score (highest).

**What changes with 4 dimensions:**
- KMO sampling adequacy may improve from 0.375 (below threshold) toward 0.6 (PCA viable)
- India's #1 ranking likely strengthens (India has highest tax + highest lending rate)

**Methodology per OECD Handbook (Nardo et al., 2008):**
- Equal-weighted CCI = mean of z-scored dimensions (primary, fallback if KMO < 0.6)
- PCA-weighted CCI computed for robustness comparison

In [ ]:
print("=" * 70)
print("Figure 6 — Composite Cost Index (4 dimensions)")
print("=" * 70)

# Load 4-dimension matrix saved in previous cell
cost_4d = pd.read_csv(CLEANED / 'rq4_cost_4dimensions_wide.csv').set_index('country')
print(f"Cost matrix shape: {cost_4d.shape}")
print(f"Countries: {cost_4d.index.tolist()}")
print(f"Dimensions: {cost_4d.columns.tolist()}")

# Drop rows with too many NaN (need at least 3 of 4 dimensions for meaningful CCI)
cost_4d_complete = cost_4d.dropna(thresh=3).copy()
print(f"\nAfter dropping rows with <3 dimensions: {len(cost_4d_complete)} countries")

# --- KMO and Bartlett diagnostics ---
print("\n--- KMO and Bartlett's diagnostics (with 4 dimensions) ---")
try:
    from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

    # Need to handle NaNs for diagnostics — use complete cases only
    cost_complete_cases = cost_4d_complete.dropna()
    print(f"  Complete cases (no NaN): {len(cost_complete_cases)} countries")

    if len(cost_complete_cases) >= 5:  # Need minimum sample for diagnostics
        chi_sq, p_value = calculate_bartlett_sphericity(cost_complete_cases)
        kmo_per_var, kmo_total = calculate_kmo(cost_complete_cases)
        print(f"  Bartlett χ² = {chi_sq:.2f}, p = {p_value:.4f}")
        print(f"  KMO total = {kmo_total:.3f}")
        print(f"  KMO per variable: {dict(zip(cost_complete_cases.columns, [round(k, 2) for k in kmo_per_var]))}")

        if kmo_total >= 0.6:
            print(f"  ✓ KMO ≥ 0.6 — PCA viable as primary methodology")
            kmo_status = "viable"
        else:
            print(f"  ⚠ KMO < 0.6 — PCA fallback to equal-weighted (per OECD Handbook)")
            kmo_status = "fallback"
    else:
        print("  ⚠ Insufficient complete cases for KMO/Bartlett")
        kmo_total = None
        kmo_status = "insufficient_data"
except ImportError:
    print("  Installing factor_analyzer...")
    import subprocess
    subprocess.run(['pip', 'install', 'factor_analyzer', '--quiet'], check=False)
    print("  → Re-run this cell after install completes")
    kmo_total = None
    kmo_status = "needs_install"

# --- Equal-weighted CCI (primary) ---
zscored = cost_4d_complete.apply(lambda c: (c - c.mean()) / c.std())
cci_equal = zscored.mean(axis=1)
cci_equal_sorted = cci_equal.sort_values(ascending=False)

print(f"\n--- Equal-weighted CCI ranking (4 dimensions) ---")
for country, score in cci_equal_sorted.items():
    print(f"  {country:12s}: {score:+.3f}")

# --- PCA-weighted CCI (robustness) ---
try:
    from sklearn.decomposition import PCA
    pca_data = cost_4d_complete.dropna()
    if len(pca_data) >= 4:
        pca = PCA(n_components=1)
        pca_zscored = pca_data.apply(lambda c: (c - c.mean()) / c.std())
        pc1 = pca.fit_transform(pca_zscored).flatten()
        cci_pca = pd.Series(pc1, index=pca_data.index)
        var_explained = pca.explained_variance_ratio_[0] * 100
        print(f"\n--- PCA-weighted CCI (PC1 explains {var_explained:.1f}% of variance) ---")
        for country, score in cci_pca.sort_values(ascending=False).items():
            print(f"  {country:12s}: {score:+.3f}")

        # Spearman correlation between equal and PCA rankings
        from scipy.stats import spearmanr
        common = cci_equal.index.intersection(cci_pca.index)
        rho, p = spearmanr(cci_equal.loc[common], cci_pca.loc[common])
        print(f"\n  Equal-weighted vs PCA-weighted Spearman ρ = {rho:.3f} (p = {p:.4f})")
        print(f"  → {'High agreement' if rho > 0.85 else 'Moderate agreement' if rho > 0.6 else 'Disagreement'} between methods")
except Exception as e:
    print(f"\n  PCA computation skipped: {e}")
    cci_pca = None

# --- Figure 6 plot ---
fig, ax = plt.subplots(figsize=(10, 5.5))
colors_bar = [CB_PALETTE['vermillion'] if c == 'India' else CB_PALETTE['sky_blue']
              for c in cci_equal_sorted.index]

bars = ax.barh(cci_equal_sorted.index, cci_equal_sorted.values,
               color=colors_bar, edgecolor='white', linewidth=0.6)

# Annotate bars
for bar, val in zip(bars, cci_equal_sorted.values):
    x_offset = 0.03 if val >= 0 else -0.03
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + x_offset, bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}', va='center', ha=ha, fontsize=9, weight='bold')

ax.axvline(0, color=CB_PALETTE['gray'], linewidth=0.8, alpha=0.6)
ax.set_xlabel('Composite Cost Index (z-score; higher = more expensive)')
ax.set_title('Composite Cost Index by Country (4 Dimensions, Equal-Weighted)\n'
             f'KMO = {kmo_total:.3f}; Lending + Real Interest + T&D Losses + Corporate Tax' if kmo_total else
             'Composite Cost Index by Country (4 Dimensions, Equal-Weighted)',
             pad=12, weight='bold')
ax.invert_yaxis()  # India (highest) at top
ax.grid(True, axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add note about methodology in bottom-left
note = ('Equal-weighted from z-scored dimensions per OECD Handbook fallback'
        if kmo_status == 'fallback' else
        'Equal-weighted; PCA-weighted also computed for robustness')
ax.text(0.02, 0.02, note, transform=ax.transAxes,
        fontsize=8, style='italic', alpha=0.7)

plt.tight_layout()
out = FIGURES / 'figure6_cci_ranking.png'
plt.savefig(out, dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: {out.name}")
plt.show()

# Save CCI for use in RQ4 modeling
cci_df = pd.DataFrame({
    'country': cci_equal.index,
    'cci_equal_4dim': cci_equal.values,
})
if cci_pca is not None:
    cci_df = cci_df.merge(
        pd.DataFrame({'country': cci_pca.index, 'cci_pca_4dim': cci_pca.values}),
        on='country', how='left',
    )
cci_df['kmo_total'] = kmo_total
cci_df['data_source'] = 'NSEMI_CCI_4dim_v2'
cci_df['retrieval_date'] = '2026-05-01'
cci_path = CLEANED / 'rq4_cci_4dim.csv'
cci_df.to_csv(cci_path, index=False)
print(f"\n✓ Saved: {cci_path.name} (CCI scores for RQ4 break-even and one-sample t-test)")

Figure 6 — Composite Cost Index (4 dimensions)
Cost matrix shape: (10, 4)
Countries: ['India', 'Vietnam', 'Germany', 'Malaysia', 'China', 'Thailand', 'S. Korea', 'Japan', 'USA', 'Singapore']
Dimensions: ['Lending Rate (%)', 'Real Interest Rate (%)', 'T&D Losses (%)', 'Corporate Tax Rate (%)']

After dropping rows with <3 dimensions: 9 countries

--- KMO and Bartlett's diagnostics (with 4 dimensions) ---
  Complete cases (no NaN): 9 countries

--- Equal-weighted CCI ranking (4 dimensions) ---
  India       : +1.243
  Vietnam     : +0.575
  Malaysia    : +0.241
  China       : +0.022
  Thailand    : -0.103
  S. Korea    : -0.171
  Japan       : -0.221
  USA         : -0.343
  Singapore   : -1.243

--- PCA-weighted CCI (PC1 explains 48.8% of variance) ---
  India       : +2.460
  Vietnam     : +1.437
  Malaysia    : +0.560
  Thailand    : +0.092
  China       : -0.044
  S. Korea    : -0.567
  Japan       : -0.755
  USA         : -0.809
  Singapore   : -2.374

  Equal-weighted vs PCA-weigh

## Cell 6 — Final inventory and next steps

In [ ]:
print("=" * 70)
print("FIGURE REGENERATION COMPLETE")
print("=" * 70)

# Inventory of all figures in folder
all_figures = sorted(FIGURES.glob('*.png'))
print(f"\nAll figures in {FIGURES.name}/:")
for f in all_figures:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.0f} KB")

print(f"\n--- Updated in this notebook ---")
updated = [
    'figure1_dgft_imports_by_hs_segment.png',
    'figure2_hhi_country_concentration.png',
    'figure5_cost_dimensions_heatmap.png',
    'figure6_cci_ranking.png',
]
for f in updated:
    p = FIGURES / f
    if p.exists():
        print(f"  ✓ {f}")
    else:
        print(f"  ⚠ NOT FOUND: {f}")

print(f"\n--- Unchanged (no underlying data change) ---")
unchanged = [
    'figure3_atc_losses_ism_vs_non_ism.png',
    'figure4_tertiary_enrollment_cross_country.png',
]
for f in unchanged:
    p = FIGURES / f
    if p.exists():
        print(f"  ✓ {f} (no regeneration needed)")
    else:
        print(f"  ⚠ NOT FOUND: {f} (check exact filename in figures/)")

print(f"\n--- Files added for RQ4 modeling ---")
modeling_files = [
    'rq4_cost_4dimensions_wide.csv',
    'rq4_cci_4dim.csv',
]
for f in modeling_files:
    p = CLEANED / f
    if p.exists():
        df = pd.read_csv(p)
        print(f"  ✓ {f}: {len(df)} rows × {len(df.columns)} cols")

print("\n--- Ready for next phase: modeling ---")
print("RQ1: regression of MOSPI IIP ~ DGFT IDR predictors (joined on year_month)")
print("RQ2: Mann-Whitney U on rq2_pfc_atc_losses.csv (no change)")
print("RQ3: Welch's t-test on rq3_unesco_graduates.csv (no change)")
print("RQ4: One-sample t-test + break-even on rq4_cci_4dim.csv (4-dim version)")

FIGURE REGENERATION COMPLETE

All figures in figures/:
  figure_01_dgft_imports_by_hs_segment.png: 390 KB
  figure_02_hhi_by_hs_segment.png: 602 KB
  figure_03_atc_losses_ism_vs_non_ism.png: 177 KB
  figure_04_tertiary_enrollment_cross_country.png: 229 KB
  figure_05_cost_dimensions_heatmap.png: 313 KB
  figure_06_composite_cost_index_by_country.png: 170 KB

--- Updated in this notebook ---
  ✓ figure_01_dgft_imports_by_hs_segment.png
  ✓ figure_02_hhi_by_hs_segment.png
  ✓ figure_05_cost_dimensions_heatmap.png
  ✓ figure_06_composite_cost_index_by_country.png

--- Unchanged (no underlying data change) ---
  ✓ figure_03_atc_losses_ism_vs_non_ism.png (no regeneration needed)
  ✓ figure_04_tertiary_enrollment_cross_country.png (no regeneration needed)

--- Files added for RQ4 modeling ---
  ✓ rq4_cost_4dimensions_wide.csv: 10 rows × 5 cols
  ✓ rq4_cci_4dim.csv: 9 rows × 6 cols

--- Ready for next phase: modeling ---
RQ1: regression of MOSPI IIP ~ DGFT IDR predictors (joined on year_month